### Pre-modeling

In [1]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv('../data/processed/Telco-Customer-Churn-Processed.csv')

In [3]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=86, stratify=y
)

---

We have to choose which metric to use.

In our business problem:
- if model makes type 1 error -> company will trigger its standard retention protocols (discounts, sending gifts etc.) basically unnecessary costs

- if model makes type 2 error -> customer will leave, which will decrease revenue of the company, it also leads to damage to reputation and to spend more money on marketing

In [5]:
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

We see that dataset is imbalanced, so accuracy can't be used there.

Precision is also not the best option because cost of false positive is not that bad.

Cost of false negative on the other side is too high, so the best option is `recall` or `f1`, but i would stick up to recall, since FN is more dangerous and `threshold` in our case would be `0.3`

---

### Model choosing (RF vs LGBM vs XGB)

---

In [6]:
# Robust to class imbalance 
ratio = (y_train == 0).sum() / (y_train == 1).sum()
THRESHOLD = 0.3

#### Random Forest 

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.50      0.66      1035
           1       0.41      0.95      0.57       374

    accuracy                           0.62      1409
   macro avg       0.69      0.73      0.62      1409
weighted avg       0.82      0.62      0.64      1409



Based on classification report we see, that model found 95% of churn customers

But on the other side it predicts only 41% of them correctly

---